# Agent State Machine (ASM) - Quick Start Guide

An **agent** is a wrapper around a finite state machine designed to accomplish a specific task and will be referred to as ASM.

A **StateMachineBuilder** is used to build the ASM from a manifest and a state diagram.

```mermaid
flowchart TD
    A["state_diagram"] -->|Input| B["StateMachineBuilder"]
    A2["state_manifest"] -->|Input| C["StateModel"]
    C -->|Build| B
    B --> D["fsm"]
```


---

## Example: Standard Agent

In this example, we will demonstrate how to create a simple **Agent** agent using a state diagram and state manifest.  
The state logic is implemented using **PureActionState** which allows custom actions to be attached to states.

### a) Define State Diagram

The following is a simple example of an **Agent** agent using 3 states:

* **INIT:** Collect initial input.
* **GENERATE:** The state where the LLM generates a response.
* **FINAL:** Return final output.

```mermaid
stateDiagram-v2
direction LR
INIT --> GENERATE: next / action
GENERATE --> FINAL: next / action
```


In [1]:
STATE_DIAGRAM = """
    INIT --> CHAT
    CHAT --> FINAL
    """

### b) Define State Manifest

The state manifest is a dictionary. 
Each state in the manifest corresponds to a state in the state diagram.


In [2]:
STATE_MANIFEST_V1 = {
    "INIT": {
        "input_data": {
            "llm_config": {"type": "getter", "dependency": "get_llm_config"}
            }
    },
    "CHAT": {
        "module_path": "gai.asm.states",
        "class_name": "PureActionState",
        "title": "CHAT",
        "action": "generate",
        "input_data": {
            "llm_config": {"type": "state_bag", "dependency": "llm_config"},
        },
        "output_data": ["streamer", "get_assistant_message"],
    },
    "FINAL": {"output_data": ["monologue"]},
}


### c) Create state action

In [3]:
from gai.dialogue.chat import AsyncOpenAI

async def generate_action(state):
    
    llm_config = state.input["llm_config"]
    client = AsyncOpenAI(llm_config)
    
    # Get message from input
    user_message = state.machine.user_message
    
    # Execute
    response = await client.chat.completions.create(
        model=llm_config["model"],
        messages=[{
            "role":"user",
            "content":user_message
            }],
        max_tokens=50,
        stream=True
    )
    
    assistant_message = ""
    async def streamer():
        nonlocal assistant_message
        async for chunk in response:
            if chunk:
                chunk = chunk.choices[0].delta.content
                if isinstance(chunk,str) and chunk:
                    assistant_message += chunk
                    yield chunk

    state.machine.state_bag["get_assistant_message"] = lambda: assistant_message
    state.machine.state_bag["streamer"] = streamer()

### c) Build State Machine

In [5]:
from gai.asm import AsyncStateMachine,  FileMonologue

with AsyncStateMachine.StateMachineBuilder(STATE_DIAGRAM) as builder:
    fsm = builder.build(
        STATE_MANIFEST_V1,
        get_llm_config=lambda state: {
            "client_type": "anthropic",
            "model": "claude-opus-4-20250514",
        },
        # get_llm_config=lambda state: {
        #     "client_type": "gai",
        #     "model": "ttt",
        #     "url": "http://gai-llm-svr:12031/gen/v1/chat/completions"
        # },
        agent_name="Sara",
        generate=generate_action,
    )
    


### d) Run State Machine (INIT->CHAT)

In [6]:
fsm.user_message = "Write a one sentence story"
await fsm.run_async()
async for chunk in fsm.state_bag["streamer"]:
    print(chunk,end='',flush=True)
print("\n\n")

The old lighthouse keeper realized, as the storm battered his tower, that the ship he was desperately trying to guide to safety was the same one that had taken his daughter away twenty years ago.




### d) Continue (CHAT->FINAL)

In [7]:
await fsm.run_async()
print("State History:")
for state in fsm.state_history:
    print(f"State: {state['state']}")
    print(f"- input: {state['input']}")
    print(f"- output: {state['output']}")
    print("-" * 20)
print("Assistant Message:")
fsm.state_bag["get_assistant_message"]()

State History:
State: INIT
- input: {'user_message': 'Write a one sentence story', 'monologue': <gai.asm.monologue.Monologue object at 0x7f4f7648f9d0>, 'step': 0, 'time': datetime.datetime(2025, 7, 9, 3, 6, 40, 95424), 'name': 'Sara', 'llm_config': {'client_type': 'anthropic', 'model': 'claude-opus-4-20250514'}}
- output: {'name': 'Sara', 'user_message': 'Write a one sentence story', 'monologue': <gai.asm.monologue.Monologue object at 0x7f4f9810e890>, 'step': 1, 'time': datetime.datetime(2025, 7, 9, 3, 6, 40, 95490)}
--------------------
State: CHAT
- input: {'name': 'Sara', 'user_message': 'Write a one sentence story', 'monologue': <gai.asm.monologue.Monologue object at 0x7f4f7648fe50>, 'step': 1, 'time': datetime.datetime(2025, 7, 9, 3, 6, 40, 95661), 'llm_config': {'client_type': 'anthropic', 'model': 'claude-opus-4-20250514'}}
- output: {'streamer': <async_generator object generate_action.<locals>.streamer at 0x7f4f7536e640>, 'get_assistant_message': <function generate_action.<loca

'The old lighthouse keeper realized, as the storm battered his tower, that the ship he was desperately trying to guide to safety was the same one that had taken his daughter away twenty years ago.'

### Complete Code Example (using ChatState and Monologue)

Here is another version of the example that replaces PureActionState with the built-in ChatState.
You can attach a FileMonologue to handle in-state memory.


In [ ]:
from gai.asm import AsyncStateMachine, FileMonologue
import os
if os.path.exists("tmp/monologue.json"):
    os.remove("tmp/monologue.json")

monologue = (
    FileMonologue(file_path="tmp/monologue.json")
)
STATE_DIAGRAM = """
    INIT --> CHAT
    CHAT --> FINAL
    """
STATE_MANIFEST_V1 = {
    "INIT": {
        "input_data": {
            "llm_config": {"type": "getter", "dependency": "get_llm_config"}
            }
    },
    "CHAT": {
        "module_path": "gai.asm.states",
        "class_name": "ChatState",
        "title": "CHAT",
        "input_data": {
            "llm_config": {"type": "state_bag", "dependency": "llm_config"},
        },
        "output_data": ["streamer", "get_assistant_message"],
    },    
    "FINAL": {"output_data": ["monologue"]},
}
with AsyncStateMachine.StateMachineBuilder(STATE_DIAGRAM) as builder:
    fsm = builder.build(
        STATE_MANIFEST_V1,
        get_llm_config=lambda state: {
            "client_type": "anthropic",
            "model": "claude-opus-4-20250514",
        },
        agent_name="Sara",
        monologue=monologue
    )
fsm.user_message = "Write a one sentence story"
await fsm.run_async()
async for chunk in fsm.state_bag["streamer"]:
    print(chunk,end='',flush=True)
print("\n\n")


The old lighthouse keeper discovered that the mysterious lights he'd been reporting for forty years were actually reflections from his own lantern, bouncing off a mirror left by his long-dead predecessor who had gone mad from loneliness.[{'citations': None, 'text': "The old lighthouse keeper discovered that the mysterious lights he'd been reporting for forty years were actually reflections from his own lantern, bouncing off a mirror left by his long-dead predecessor who had gone mad from loneliness.", 'type': 'text'}]




## Check Monologue

In [ ]:
with open("tmp/monologue.json","r") as f:
    txt = f.read()
print(txt)

[
    {
        "id": "e57abb3c-eae5-473d-be25-0205c3cede6a",
        "header": {
            "sender": "User",
            "recipient": "Assistant",
            "timestamp": 1752030208.8509426,
            "order": 0
        },
        "body": {
            "type": "state",
            "state_name": "",
            "step_no": -1,
            "content_type": "text",
            "role": "user",
            "content": "Write a one sentence story"
        }
    },
    {
        "id": "d535c3d9-4048-49a0-986f-d038ff0c70aa",
        "header": {
            "sender": "Assistant",
            "recipient": "User",
            "timestamp": 1752030211.723532,
            "order": 0
        },
        "body": {
            "type": "state",
            "state_name": "",
            "step_no": -1,
            "content_type": "text",
            "role": "assistant",
            "content": [
                {
                    "citations": null,
                    "text": "The old lighthouse keepe

---

## Example: Standard Agent (Part 2)

Same example but with an additional state to demonstrate context management by monologue messages.

```mermaid
stateDiagram-v2
direction LR
INIT --> CHAT
CHAT --> CONTINUE
CONTINUE --> FINAL
```


In [3]:
from gai.dialogue.chat import AsyncOpenAI
from gai.asm import AsyncStateMachine, FileMonologue
import os
if os.path.exists("tmp/monologue.json"):
    os.remove("tmp/monologue.json")

monologue = FileMonologue(file_path="tmp/monologue.json")


STATE_DIAGRAM = """
    INIT --> CHAT
    CHAT --> CONTINUE
    CONTINUE --> FINAL
    """
    
STATE_MANIFEST_V1 = {
    "INIT": {
        "input_data": {
            "llm_config": {"type": "getter", "dependency": "get_llm_config"},
        }
    },
    "CHAT": {
        "module_path": "gai.asm.states",
        "class_name": "ChatState",
        "title": "CHAT",
        "input_data": {
            "llm_config": {"type": "state_bag", "dependency": "llm_config"},
        },
        "output_data": ["streamer", "get_assistant_message"],
    },
    "CONTINUE": {
        "module_path": "gai.asm.states",
        "class_name": "PureActionState",
        "title": "CONTINUE",
        "action": "continue_action",
        "input_data": {
            "llm_config": {"type": "state_bag", "dependency": "llm_config"},
        },
        "output_data": ["streamer", "get_assistant_message"],
    },
    "FINAL": {"output_data": ["monologue"]},
}

async def continue_action(state):
    
    llm_config = state.input["llm_config"]
    client = AsyncOpenAI(llm_config)
    
    # Import data from state_bag
    state.machine.monologue.add_user_message(state=state,content="Please continue.")
    
    # Execute
    
    response = await client.chat.completions.create(
        model=llm_config["model"],
        messages=state.machine.monologue.list_chat_messages(),
        max_tokens=50,
        stream=True
    )
    
    assistant_message = ""
    async def streamer():
        nonlocal assistant_message
        async for chunk in response:
            if chunk:
                chunk = chunk.choices[0].delta.content
                if isinstance(chunk,str) and chunk:
                    assistant_message += chunk
                    yield chunk
        state.machine.monologue.add_assistant_message(
            state=state, content=assistant_message
        )
        # Need to update the stale history due to delayed output
        state.machine.state_history[-1]["output"]["monologue"] = (
            state.machine.monologue.copy()
        )
    
    state.machine.state_bag["get_assistant_message"] = lambda: assistant_message
    state.machine.state_bag["streamer"] = streamer()
    
from gai.asm import AsyncStateMachine

with AsyncStateMachine.StateMachineBuilder(STATE_DIAGRAM) as builder:
    fsm = builder.build(
        STATE_MANIFEST_V1,
        get_llm_config=lambda state: {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
        },
        monologue=monologue,        
        agent_name="Sara",
        continue_action=continue_action,
    )


### a) Run State Machine (INIT->CHAT)

In [4]:
fsm.user_message = "Write a one sentence story"
await fsm.run_async()
async for chunk in fsm.state_bag["streamer"]:
    print(chunk,end='',flush=True)
print("\n\n")

print("Step 1:")
print("Before:")
for message in fsm.state_history[1]["input"]["monologue"].list_messages():
    print(
        f"{message.header.timestamp} {message.header.sender} > {message.body.content}"
    )

print("After:")
for message in fsm.state_history[1]["output"]["monologue"].list_messages():
    print(
        f"{message.header.timestamp} {message.header.sender} > {message.body.content}"
    )

The last librarian on Earth carefully shelved the final book as the automated city hummed emptily around her, whispering "Until we meet again" to the silent stacks before stepping into the golden light that would carry her to join the others among the stars.[{'citations': None, 'text': 'The last librarian on Earth carefully shelved the final book as the automated city hummed emptily around her, whispering "Until we meet again" to the silent stacks before stepping into the golden light that would carry her to join the others among the stars.', 'type': 'text'}]


Step 1:
Before:
After:


### c) Run State Machine (GENERATE->CONTINUE)

In [5]:
await fsm.run_async()
async for chunk in fsm.state_bag["streamer"]:
    print(chunk,end='',flush=True)
print("\n\n")

print("Step 2:")
print("Before:")
for message in fsm.state_history[2]["input"]["monologue"].list_messages():
    print(
        f"{message.header.timestamp} {message.header.sender} > {message.body.content}"
    )

print("After:")
for message in fsm.state_history[2]["output"]["monologue"].list_messages():
    print(
        f"{message.header.timestamp} {message.header.sender} > {message.body.content}"
    )

The book—a worn copy of *The Velveteen Rabbit*—glowed softly as her fingers left its spine, and in that moment the entire library shimmered and folded itself into a single point of light that shot


Step 2:
Before:
After:


### e) END (GENERATE->FINAL)

In [6]:
await fsm.run_async()
print("State History:")
for state in fsm.state_history:
    print(f"State: {state['state']}")
    print(f"- input: {state['input']}")
    print(f"- output: {state['output']}")
    print("-" * 20)
print("Assistant Message:")
fsm.state_bag["get_assistant_message"]()

State History:
State: INIT
- input: {'user_message': 'Write a one sentence story', 'monologue': <gai.asm.monologue.FileMonologue object at 0x71fa2e26c8b0>, 'step': 0, 'time': datetime.datetime(2025, 7, 9, 3, 10, 39, 988166), 'name': 'Sara', 'llm_config': {'client_type': 'anthropic', 'model': 'claude-sonnet-4-20250514'}}
- output: {'name': 'Sara', 'user_message': 'Write a one sentence story', 'monologue': <gai.asm.monologue.FileMonologue object at 0x71fa4c2caa40>, 'step': 1, 'time': datetime.datetime(2025, 7, 9, 3, 10, 39, 988265)}
--------------------
State: CHAT
- input: {'name': 'Sara', 'user_message': 'Write a one sentence story', 'monologue': <gai.asm.monologue.FileMonologue object at 0x71fa2e26cbb0>, 'step': 1, 'time': datetime.datetime(2025, 7, 9, 3, 10, 39, 988537), 'llm_config': {'client_type': 'anthropic', 'model': 'claude-sonnet-4-20250514'}}
- output: {'streamer': <async_generator object ChatState.run_async.<locals>.streamer at 0x71fa2de1ecc0>, 'name': 'Sara', 'user_message'

'The book—a worn copy of *The Velveteen Rabbit*—glowed softly as her fingers left its spine, and in that moment the entire library shimmered and folded itself into a single point of light that shot'